# PincOpen as a Unitree Dex1: tests

Checks the motor, the calibration, and the Dex1 DDS interface of `pincopen_dex1_server.py`.
The bridge runs inside this notebook (in a background thread), and the notebook plays the role
of a Dex1 client such as `xr_teleoperate`, so no second terminal is needed.

**Before you start**
- Kernel: **PincOpen Dex1 (py3.10)**.
- Shut down any other kernel that has the serial port open (e.g. `feetech_test.ipynb`).
- Cells marked ⚠️ move the gripper. Keep fingers out of it.

In [ ]:
import logging, math, os, pathlib, subprocess, sys, threading, time

BRIDGE_DIR = (pathlib.Path.cwd() / "../dex1_bridge").resolve()  # this notebook lives in notebooks/
sys.path.insert(0, str(BRIDGE_DIR))
import feetech
from feetech import STS3215, describe_error
from pincopen_dex1_server import Bridge, Calibration, CMD_TIMEOUT_S, Q_OPEN

logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("pincopen_dex1").setLevel(logging.INFO)

PORT = "/dev/cu.usbmodem58CD1773071"
SIDE = "right"     # rt/dex1/right/* or rt/dex1/left/*
NETWORK = None     # DDS interface; None = auto, "eth0" on the robot network

cal = Calibration.load(BRIDGE_DIR / "calibration.json")
holders = subprocess.run(["lsof", "-t", PORT], capture_output=True, text=True).stdout.split()
holders = [pid for pid in holders if pid != str(os.getpid())]
if holders:
    print(f"WARNING: {PORT} is also open by process {', '.join(holders)}. "
          "Shut down other kernels (e.g. feetech_test.ipynb) or servers before continuing.")

print(f"calibration: motor id {cal.motor_id}, closed raw {cal.closed_raw}, open raw {cal.open_raw}")

results = {}
def check(name, ok, detail=""):
    results[name] = bool(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {name}  {detail}")

## 1. Motor (serial only, no DDS)

### 1.1 Connection and health

In [ ]:
if "bridge_thread" in globals() and bridge_thread.is_alive():
    raise RuntimeError("The bridge is running: run section 3 first.")
if "motor" in globals():
    motor.close()  # re-running this cell: release the port held by the previous connection
motor = STS3215(PORT, cal.motor_id)
check("motor answers on its ID", motor.ping(), f"id {cal.motor_id}")

raw, speed, load = motor.read_state()
volts = motor.read(feetech.PRESENT_VOLTAGE, 1)[0] / 10
temp = motor.read_temperature()
check("supply voltage 10.5-13 V", 10.5 <= volts <= 13, f"{volts:.1f} V")
check("temperature below 50 C", temp < 50, f"{temp} C")
check("no motor error flags", motor.last_error == 0, describe_error(motor.last_error))
print(f"position: raw {raw} -> q = {cal.q_from_raw(raw):.3f} rad")

### 1.2 Bus speed
The bridge loop runs at 200 Hz; each loop does one read and at most one write.

In [ ]:
n = 200
t0 = time.perf_counter()
for _ in range(n):
    motor.read_state()
rate = n / (time.perf_counter() - t0)
check("state read faster than 500 Hz", rate > 500, f"{rate:.0f} Hz")

### 1.3 Calibration (by hand, torque off)
Run the cell, then **open the gripper fully and close it again** by hand. The cell finishes as soon
as it has seen both ends (or after 30 s).

In [ ]:
motor.set_torque(False)
qs, seen_open = [], False
t_end = time.monotonic() + 30
while time.monotonic() < t_end:
    q = cal.q_from_raw(motor.read_state()[0])
    qs.append(q)
    seen_open = seen_open or q > Q_OPEN - 0.3
    print(f"\rq = {q:6.3f} rad   min {min(qs):6.3f}   max {max(qs):6.3f}   ({t_end - time.monotonic():4.1f} s left)", end="")
    if seen_open and q < 0.3:  # opened fully, then closed again
        break
    time.sleep(0.02)
print()
if max(qs) - min(qs) < 1.0:
    print("SKIP  the gripper was not moved by hand: re-run this cell and open/close it while it counts down")
else:
    check("closed reads q ~ 0", abs(min(qs)) < 0.2, f"min {min(qs):.3f}")
    check(f"open reads q ~ {Q_OPEN}", abs(max(qs) - Q_OPEN) < 0.2, f"max {max(qs):.3f}")
    print("If these fail, re-run `python pincopen_dex1_server.py --calibrate` in dex1_bridge/ and re-run the setup cell.")

### 1.4 ⚠️ Direct position moves
Moves the gripper with plain serial commands (no DDS): half open, open, closed.

In [ ]:
def move_to(q, timeout=3.0, tol=0.1):
    motor.set_goal_position(cal.raw_from_q(q))
    t0 = time.monotonic()
    while time.monotonic() - t0 < timeout:
        now = cal.q_from_raw(motor.read_state()[0])
        if abs(now - q) < tol:
            return now, time.monotonic() - t0
        time.sleep(0.01)
    return now, None

motor.set_goal_position(motor.read_state()[0])  # no jump when torque turns on
motor.set_torque(True)
for target in (Q_OPEN / 2, Q_OPEN, 0.0):
    reached, elapsed = move_to(target)
    check(f"reaches q = {target:.1f}", elapsed is not None,
          f"at {reached:.3f} rad" + (f" in {elapsed:.2f} s" if elapsed else " (timed out)"))
motor.set_torque(False)

## 2. Dex1 DDS interface

### 2.1 Start the bridge and a Dex1 client
The bridge uses the same `motor` object (its serial access is locked, so the notebook can still
read the motor while the bridge runs).

In [ ]:
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.default import unitree_go_msg_dds__MotorCmd_
from unitree_sdk2py.idl.unitree_go.msg.dds_ import MotorCmds_, MotorStates_

if "bridge_thread" in globals() and bridge_thread.is_alive():
    print("Bridge already running, not starting a second one (run section 3 to stop it).")
else:
    ChannelFactoryInitialize(0, NETWORK)  # no-op if already initialized in this kernel

    # Client side first, like xr_teleoperate's Dex1_1_Gripper_Controller, so the reader exists
    # before the bridge publishes its first state message.
    latest = {"n": 0, "state": None}
    def on_state(msg):
        if msg.states:
            latest["state"] = msg.states[0]
            latest["n"] += 1

    state_sub = ChannelSubscriber(f"rt/dex1/{SIDE}/state", MotorStates_)
    state_sub.Init(on_state, 10)
    cmd_pub = ChannelPublisher(f"rt/dex1/{SIDE}/cmd", MotorCmds_)
    cmd_pub.Init()
    cmd = MotorCmds_(cmds=[unitree_go_msg_dds__MotorCmd_()])
    cmd.cmds[0].mode, cmd.cmds[0].kp, cmd.cmds[0].kd = 1, 5.0, 0.05

    bridge = Bridge(motor, cal, SIDE, rate_hz=200)
    bridge_thread = threading.Thread(target=bridge.run, daemon=True)
    bridge_thread.start()

def send(q):
    cmd.cmds[0].q = q
    cmd_pub.Write(cmd)

def state_q():
    return latest["state"].q

def command(q, duration=1.5, rate=100):
    """Publish q continuously for `duration` seconds, return the reported state q."""
    t_end = time.monotonic() + duration
    while time.monotonic() < t_end:
        send(q)
        time.sleep(1 / rate)
    return state_q()

time.sleep(1.0)
check("bridge thread running", bridge_thread.is_alive())

### 2.2 State is published without any command
xr_teleoperate waits for state before it sends anything.

In [ ]:
n0 = latest["n"]
time.sleep(2.0)
rate = (latest["n"] - n0) / 2.0
check("state published before any command", latest["n"] > 0)
check("state rate at least 100 Hz", rate >= 100, f"{rate:.0f} Hz")
motor_q = cal.q_from_raw(motor.read_state()[0])
check("state q matches the motor", abs(state_q() - motor_q) < 0.05, f"state {state_q():.3f} vs motor {motor_q:.3f}")
check("torque still off (no command yet)", motor.read(feetech.TORQUE_ENABLE, 1)[0] == 0)

### 2.3 ⚠️ Position commands

In [ ]:
for target in (Q_OPEN / 2, Q_OPEN, 0.0):
    q = command(target)
    check(f"cmd q = {target:.1f} reached", abs(q - target) < 0.1, f"state q {q:.3f}")

### 2.4 ⚠️ Out-of-range commands are clamped

In [ ]:
q = command(9.0)
check(f"q = 9.0 clamps to {Q_OPEN}", abs(q - Q_OPEN) < 0.1, f"state q {q:.3f}")
q = command(-2.0)
check("q = -2.0 clamps to 0", abs(q) < 0.1, f"state q {q:.3f}")

### 2.5 ⚠️ xr_teleoperate-style control
xr_teleoperate's `Dex1_1_Gripper_Controller` runs at 200 Hz and clips every command to within
±0.18 rad of the reported state, so the gripper only moves if the state feedback is live and accurate.

In [ ]:
DELTA_GRIPPER_CMD = 0.18

def xr_ramp(target, timeout=6.0, fps=200):
    t0 = time.monotonic()
    while time.monotonic() - t0 < timeout:
        s = state_q()
        if abs(s - target) < 0.1:
            return time.monotonic() - t0
        send(min(max(target, s - DELTA_GRIPPER_CMD), s + DELTA_GRIPPER_CMD))
        time.sleep(1 / fps)
    return None

for target in (Q_OPEN, 0.0):
    elapsed = xr_ramp(target)
    check(f"xr-style ramp to q = {target:.1f}", elapsed is not None,
          f"{elapsed:.2f} s" if elapsed else f"stuck at {state_q():.3f}")

### 2.6 ⚠️ Command timeout: hold position
After 1 s without commands the bridge holds where it is (the Dex1 brakes).

In [ ]:
command(Q_OPEN / 2)
q_before = state_q()
time.sleep(CMD_TIMEOUT_S + 1.0)
torque = motor.read(feetech.TORQUE_ENABLE, 1)[0]
check("torque stays on after timeout", torque == 1)
check("position held after timeout", abs(state_q() - q_before) < 0.1, f"{q_before:.3f} -> {state_q():.3f}")
q = command(0.0)
check("resumes on the next command", abs(q) < 0.1, f"state q {q:.3f}")

### 2.7 ⚠️ Unitree sine sweep
The same motion as `test_dex1_1_gripper_server`: q = 3.0 ± 2.5 rad, 3 s period, two periods.
Measures the delay between command and state, and how well the state follows the command's
shape once that delay is removed.

In [ ]:
CENTER, AMPLITUDE, PERIOD, CYCLES = 3.0, 2.5, 3.0, 2

command(CENTER - AMPLITUDE, duration=1.0)  # start at the bottom of the sine
log_t, log_cmd, log_q = [], [], []
t0 = time.monotonic()
while (t := time.monotonic() - t0) < CYCLES * PERIOD:
    q_cmd = CENTER - AMPLITUDE * math.cos(2 * math.pi * t / PERIOD)
    send(q_cmd)
    log_t.append(t); log_cmd.append(min(max(q_cmd, 0.0), Q_OPEN)); log_q.append(state_q())
    time.sleep(0.005)
command(0.0)

import numpy as np
t, c, s = np.array(log_t), np.array(log_cmd), np.array(log_q)
steady = t > 0.5
shifts = np.arange(0.0, 0.5, 0.005)
rms_at = [np.sqrt(np.mean((np.interp(t[steady] - d, t, c) - s[steady]) ** 2)) for d in shifts]
best = int(np.argmin(rms_at))
lag, shape_rms = shifts[best], rms_at[best]
raw_rms = np.sqrt(np.mean((c - s) ** 2))
print(f"{len(t)} samples: lag {lag * 1000:.0f} ms, shape error {shape_rms:.3f} rad rms "
      f"(error without removing the lag: {raw_rms:.3f} rad rms)")
check("sweep covers the full stroke", s.max() > Q_OPEN - 0.3 and s.min() < CENTER - AMPLITUDE + 0.3,
      f"state {s.min():.2f}..{s.max():.2f} rad")
check("sweep lag below 300 ms", lag < 0.3, f"{lag * 1000:.0f} ms")
check("sweep shape error below 0.2 rad rms", shape_rms < 0.2, f"{shape_rms:.3f} rad")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 3))
    plt.plot(log_t, log_cmd, label="command q")
    plt.plot(log_t, log_q, label="state q")
    plt.xlabel("time (s)"); plt.ylabel("q (rad)"); plt.legend(); plt.grid(alpha=0.3)
    plt.show()
except ImportError:
    pass

### 2.8 ⚠️ Grip an object (manual)
Hold an object between the fingers, run the cell: it closes for 4 s while printing the estimated
torque and motor flags, then opens. An `overload` flag means the motor's protection kicked in.

In [ ]:
t_end = time.monotonic() + 4.0
while time.monotonic() < t_end:
    send(0.0)
    s = latest["state"]
    print(f"\rq {s.q:6.3f}  tau_est {s.tau_est:6.3f} Nm  flags: {describe_error(motor.last_error):<20}", end="")
    time.sleep(0.01)
print()
command(Q_OPEN / 2)

## 3. Stop and summary

In [ ]:
bridge.running = False
bridge_thread.join(timeout=2.0)
motor.set_torque(False)
motor.close()

passed = sum(results.values())
print(f"{passed}/{len(results)} checks passed")
for name, ok in results.items():
    if not ok:
        print("  FAIL", name)